In [ ]:
!pip install micom tqdm cobra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 11.7 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
from micom import Community
from micom.workflows import build
from micom import load_pickle
from micom.workflows import grow
from micom.workflows import tradeoff
from micom.workflows import complete_community_medium
from micom.interaction import interactions
from micom.interaction import summarize_interactions
from micom.interaction import MES
from micom.measures import production_rates
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/path_to_file/community_micom_complete.csv')
data

,id,genus,species,reactions,metabolites,file,sample_id,abundance
0,mcaps,Methylococcus,Methylococcus capsulatus Bath,915,880,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_1_1,1000
1,agitata,Dechloromonas,Dechloromonas agitata is5,1770,1439,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_1_1,0
2,mcaps,Methylococcus,Methylococcus capsulatus Bath,915,880,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_1_2,900
3,agitata,Dechloromonas,Dechloromonas agitata is5,1770,1439,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_1_2,100
4,mcaps,Methylococcus,Methylococcus capsulatus Bath,915,880,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_1_3,800
...,...,...,...,...,...,...,...,...
83,agitata,Dechloromonas,Dechloromonas agitata is5,1770,1439,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_4_9,800
84,ob3,Methylosinus,Methylosinus trichosporium OB3b,1048,1020,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_4_10,100
85,agitata,Dechloromonas,Dechloromonas agitata is5,1770,1439,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_4_10,900
86,ob3,Methylosinus,Methylosinus trichosporium OB3b,1048,1020,/content/drive/MyDrive/sup_notebooks/final_mod...,sample_4_11,0


In [ ]:
import os
import pandas as pd
from micom import Community

out_folder = "path/"
os.makedirs(out_folder, exist_ok=True)

data["file"] = data["file"].str.strip()
data["id"] = data["id"].str.strip()

for sample_id, sample_data in data.groupby("sample_id"):
    out_path = os.path.join(out_folder, f"{sample_id}.pickle")

    if os.path.exists(out_path):
        print(f"Skipping {sample_id}, already exists.")
        continue

    sample_data = sample_data.reset_index(drop=True)

    print(f"Building community for {sample_id}...")
    print(sample_data[["id", "file"]])

    com = Community(sample_data, progress=False)
    com.to_pickle(out_path)
    print(f"Saved: {out_path}")

Skipping sample_1_1, already exists.
Skipping sample_1_10, already exists.
Skipping sample_1_11, already exists.
Skipping sample_1_2, already exists.
Skipping sample_1_3, already exists.
Skipping sample_1_4, already exists.
Skipping sample_1_5, already exists.
Skipping sample_1_6, already exists.
Skipping sample_1_7, already exists.
Skipping sample_1_8, already exists.
Skipping sample_1_9, already exists.
Skipping sample_2_1, already exists.
Skipping sample_2_10, already exists.
Skipping sample_2_11, already exists.
Skipping sample_2_2, already exists.
Skipping sample_2_3, already exists.
Skipping sample_2_4, already exists.
Skipping sample_2_5, already exists.
Skipping sample_2_6, already exists.
Skipping sample_2_7, already exists.
Skipping sample_2_8, already exists.
Skipping sample_2_9, already exists.
Skipping sample_3_1, already exists.
Skipping sample_3_10, already exists.
Skipping sample_3_11, already exists.
Skipping sample_3_2, already exists.
Skipping sample_3_3, already exi

In [ ]:
import glob
from micom import load_pickle

out_folder = "/content/drive/MyDrive/path_to_file/"

pickle_files = glob.glob(os.path.join(out_folder, "*.pickle"))

records = []

for pickle_path in sorted(pickle_files):
    sample_id = os.path.basename(pickle_path).replace(".pickle", "")

    com = load_pickle(pickle_path)

    n_reactions = len(com.reactions)
    n_metabolites = len(com.metabolites)

    file_name = os.path.basename(pickle_path)

    records.append({
        "reactions": n_reactions,
        "metabolites": n_metabolites,
        "file": file_name,
        "sample_id": sample_id
    })

manifest_df = pd.DataFrame(records)

manifest_path = os.path.join(out_folder, "manifest.csv")
manifest_df.to_csv(manifest_path, index=False)

manifest_df

,reactions,metabolites,file,sample_id
0,964,871,sample_1_1.pickle,sample_1_1
1,2840,2416,sample_1_10.pickle,sample_1_10
2,1918,1587,sample_1_11.pickle,sample_1_11
3,2840,2416,sample_1_2.pickle,sample_1_2
4,2840,2416,sample_1_3.pickle,sample_1_3
5,2840,2416,sample_1_4.pickle,sample_1_4
6,2840,2416,sample_1_5.pickle,sample_1_5
7,2840,2416,sample_1_6.pickle,sample_1_6
8,2840,2416,sample_1_7.pickle,sample_1_7
9,2840,2416,sample_1_8.pickle,sample_1_8


In [ ]:
manifest_df = pd.read_csv('/content/drive/MyDrive/path_to_file/manifest.csv')
manifest_df

,reactions,metabolites,file,sample_id
0,964,871,sample_1_1.pickle,sample_1_1
1,2840,2416,sample_1_10.pickle,sample_1_10
2,1918,1587,sample_1_11.pickle,sample_1_11
3,2840,2416,sample_1_2.pickle,sample_1_2
4,2840,2416,sample_1_3.pickle,sample_1_3
5,2840,2416,sample_1_4.pickle,sample_1_4
6,2840,2416,sample_1_5.pickle,sample_1_5
7,2840,2416,sample_1_6.pickle,sample_1_6
8,2840,2416,sample_1_7.pickle,sample_1_7
9,2840,2416,sample_1_8.pickle,sample_1_8


In [ ]:
from micom.workflows import grow

out_folder = '/content/drive/MyDrive/path_to_file/'

data["id"] = data["id"].str.strip()

methanotrophs = {
    "mcaps": 18.46,
    "zr20": 11.7,
    "album": 8.0,
    "ob3": 14.9
}

sample_meth_flux = {}
for sample_id, group in data.groupby("sample_id"):
    meth_id = None
    for org_id in group["id"]:
        if org_id in methanotrophs:
            meth_id = org_id
            break
    if meth_id is None:
        raise ValueError(f"methanotroph model not found {sample_id}")
    sample_meth_flux[sample_id] = methanotrophs[meth_id]

base_medium = {
    "EX_ca2_m": 1000.0,
    "EX_cd2_m": 1000.0,
    "EX_cl_m": 1000.0,
    "EX_co_m": 1000.0,
    "EX_cobalt2_m": 1000.0,
    "EX_cu2_m": 1000.0,
    "EX_fe2_m": 1000.0,
    "EX_h2o_m": 1000.0,
    "EX_h_m": 1000.0,
    "EX_k_m": 1000.0,
    "EX_mg2_m": 1000.0,
    "EX_mn2_m": 1000.0,
    "EX_na1_m": 1000.0,
    "EX_ni2_m": 1000.0,
    "EX_nh3_m": 1000.0,
    "EX_pi_m": 1000.0,
    "EX_so4_m": 1000.0,
    "EX_zn2_m": 1000.0,
    "EX_fe3_m": 1000.0,
    "EX_cro4_m": 1000.0,
    "EX_mobd_m": 1000.0,
    "EX_hg2_m": 1000.0,
    "EX_aso4_m": 1000.0,
    "EX_co2_m": 1000.0,
    "EX_h2_m": 1000.0,
    "EX_h2s_m": 1000.0,
    "EX_o2_m": 1000.0,
    "EX_clo4_m": 1000.0
}

all_results = []

for sample_id in data["sample_id"].unique():

    pickle_path = os.path.join(out_folder, f"{sample_id}.pickle")
    com = load_pickle(pickle_path)

    taxa = list(com.taxa)
    if len(taxa) != 2:
        continue

    medium = base_medium.copy()
    medium["EX_ch4_m"] = sample_meth_flux[sample_id]

    try:
        com.objective.expression
        com.medium = medium
        sol = com.optimize()

        if sol is not None and sol.status == "optimal":
            growth_rate = sol.growth_rate

            growth_rates = sol.members['growth_rate']
            growth1 = growth_rates.get(taxa[0], None)
            growth2 = growth_rates.get(taxa[1], None)

            all_results.append({
                "sample_id": sample_id,
                "taxon1": taxa[0],
                "taxon2": taxa[1],
                "growth_rate": growth_rate,
                "growth_rate1": growth1,
                "growth_rate2": growth2,
                "status": sol.status
            })
        else:
            print(f"{sample_id}: {sol.status if sol else 'None'}")
            all_results.append({
                "sample_id": sample_id,
                "taxon1": taxa[0],
                "taxon2": taxa[1],
                "growth_rate": None,
                "growth_rate1": None,
                "growth_rate2": None,
                "status": sol.status if sol else "no_solution"
            })
    except Exception as e:
        print(f"Ошибка для {sample_id}: {e}")
        all_results.append({
            "sample_id": sample_id,
            "taxon1": taxa[0],
            "taxon2": taxa[1],
            "growth_rate": None,
            "growth_rate1": None,
            "growth_rate2": None,
            "status": f"error: {str(e)[:50]}"
        })

In [ ]:
results_df = pd.DataFrame(all_results)
results_df

,sample_id,taxon1,taxon2,growth_rate,growth_rate1,growth_rate2,status
0,sample_1_2,mcaps,agitata,0.423475,0.470528,0.000000,optimal
1,sample_1_3,mcaps,agitata,0.504139,0.630174,0.000000,optimal
2,sample_1_4,mcaps,agitata,0.570519,0.815028,0.000000,optimal
3,sample_1_5,mcaps,agitata,0.634745,1.057909,0.000000,optimal
4,sample_1_6,mcaps,agitata,0.698971,1.397943,0.000000,optimal
5,sample_1_7,mcaps,agitata,0.762164,1.905409,0.000000,optimal
6,sample_1_8,mcaps,agitata,0.813828,2.549656,0.069902,optimal
7,sample_1_9,mcaps,agitata,0.673795,1.331301,0.509418,optimal
8,sample_1_10,mcaps,agitata,0.395262,0.000000,0.439180,optimal
9,sample_2_2,zr20,agitata,0.280442,0.043084,2.416659,optimal


In [ ]:
results_df.to_csv('/content/drive/MyDrive/path_to_file/com_growth_on_medium.csv')

In [ ]:
manifest_df = manifest_df[manifest_df["sample_id"].isin(results_df["sample_id"])]
manifest_df

,reactions,metabolites,file,sample_id
1,2840,2416,sample_1_10.pickle,sample_1_10
3,2840,2416,sample_1_2.pickle,sample_1_2
4,2840,2416,sample_1_3.pickle,sample_1_3
5,2840,2416,sample_1_4.pickle,sample_1_4
6,2840,2416,sample_1_5.pickle,sample_1_5
7,2840,2416,sample_1_6.pickle,sample_1_6
8,2840,2416,sample_1_7.pickle,sample_1_7
9,2840,2416,sample_1_8.pickle,sample_1_8
10,2840,2416,sample_1_9.pickle,sample_1_9
12,2369,2018,sample_2_10.pickle,sample_2_10


In [ ]:
manifest_df.to_csv('/content/drive/MyDrive/path_to_file/manifest.csv')

In [ ]:
manifest_df

,reactions,metabolites,file,sample_id
1,2840,2416,sample_1_10.pickle,sample_1_10
3,2840,2416,sample_1_2.pickle,sample_1_2
4,2840,2416,sample_1_3.pickle,sample_1_3
5,2840,2416,sample_1_4.pickle,sample_1_4
6,2840,2416,sample_1_5.pickle,sample_1_5
7,2840,2416,sample_1_6.pickle,sample_1_6
8,2840,2416,sample_1_7.pickle,sample_1_7
9,2840,2416,sample_1_8.pickle,sample_1_8
10,2840,2416,sample_1_9.pickle,sample_1_9
12,2369,2018,sample_2_10.pickle,sample_2_10


In [ ]:
manifest_df = pd.read_csv('/content/drive/MyDrive/path_to_file/manifest.csv')

In [ ]:
results_df = pd.read_csv('/content/drive/MyDrive/path_to_file/com_growth_on_medium.csv')

In [ ]:
import numpy as np

results_df = results_df[
    ~((results_df['growth_rate1'].isna()) | (results_df['growth_rate1'] == 0) |
      (results_df['growth_rate2'].isna()) | (results_df['growth_rate2'] == 0))
]
results_df

,Unnamed: 0,sample_id,taxon1,taxon2,growth_rate,growth_rate1,growth_rate2,status
6,6,sample_1_8,mcaps,agitata,0.813828,2.549656,0.069902,optimal
7,7,sample_1_9,mcaps,agitata,0.673795,1.331301,0.509418,optimal
9,9,sample_2_2,zr20,agitata,0.280442,0.043084,2.416659,optimal
10,10,sample_2_3,zr20,agitata,0.340607,0.052844,1.491657,optimal
18,18,sample_3_2,album,agitata,0.936059,0.687874,3.169723,optimal
19,19,sample_3_3,album,agitata,0.903169,0.342571,3.145560,optimal


In [ ]:
manifest_df = manifest_df[manifest_df["sample_id"].isin(results_df["sample_id"])]
manifest_df

,Unnamed: 0,reactions,metabolites,file,sample_id
1,1,2840,2416,sample_1_10.pickle,sample_1_10
6,6,2840,2416,sample_1_5.pickle,sample_1_5
7,7,2840,2416,sample_1_6.pickle,sample_1_6
8,8,2840,2416,sample_1_7.pickle,sample_1_7
15,15,2369,2018,sample_2_3.pickle,sample_2_3
16,16,2369,2018,sample_2_4.pickle,sample_2_4
17,17,2369,2018,sample_2_5.pickle,sample_2_5
18,18,2369,2018,sample_2_6.pickle,sample_2_6
25,25,3300,2987,sample_3_2.pickle,sample_3_2
26,26,3300,2987,sample_3_3.pickle,sample_3_3


In [ ]:
from micom.workflows import grow
import os
import pandas as pd

data["id"] = data["id"].str.strip()

methanotrophs = {
    "mcaps": 18.46,
    "zr20": 11.7,
    "album": 8.0,
    "ob3": 14.9
}

sample_meth_flux = {}
for sample_id, group in data.groupby("sample_id"):
    meth_id = None
    for org_id in group["id"]:
        if org_id in methanotrophs:
            meth_id = org_id
            break
    if meth_id is None:
        raise ValueError(f"methanotroph model not found {sample_id}")
    sample_meth_flux[sample_id] = methanotrophs[meth_id]

base_medium = {
    "EX_ca2_m": 1000.0,
    "EX_cd2_m": 1000.0,
    "EX_cl_m": 1000.0,
    "EX_co_m": 1000.0,
    "EX_cobalt2_m": 1000.0,
    "EX_cu2_m": 1000.0,
    "EX_fe2_m": 1000.0,
    "EX_h2o_m": 1000.0,
    "EX_h_m": 1000.0,
    "EX_k_m": 1000.0,
    "EX_mg2_m": 1000.0,
    "EX_mn2_m": 1000.0,
    "EX_na1_m": 1000.0,
    "EX_ni2_m": 1000.0,
    "EX_nh3_m": 1000.0,
    "EX_pi_m": 1000.0,
    "EX_so4_m": 1000.0,
    "EX_zn2_m": 1000.0,
    "EX_fe3_m": 1000.0,
    "EX_cro4_m": 1000.0,
    "EX_mobd_m": 1000.0,
    "EX_hg2_m": 1000.0,
    "EX_aso4_m": 1000.0,
    "EX_co2_m": 1000.0,
    "EX_h2_m": 1000.0,
    "EX_h2s_m": 1000.0,
    "EX_clo4_m": 1000.0
}

medium_records = []
for sample_id in manifest_df["sample_id"]:
    ch4_flux = sample_meth_flux[sample_id]
    med = base_medium.copy()
    med["EX_ch4_m"] = ch4_flux
    for reaction, flux in med.items():
        medium_records.append({
            "sample_id": sample_id,
            "reaction": reaction,
            "flux": flux
        })

medium_df = pd.DataFrame(medium_records)

output_dir = "/content/drive/MyDrive/path_to_file/grow_results_micom"
os.makedirs(output_dir, exist_ok=True)

for sample_id in manifest_df["sample_id"].unique():
    print(f"Обработка образца {sample_id}")

    manifest_subset = manifest_df[manifest_df["sample_id"] == sample_id].copy()

    medium_subset = medium_df[medium_df["sample_id"] == sample_id].copy()

    res = grow(
        manifest_subset,
        model_folder="/content/drive/MyDrive/path_to_file/",
        medium=medium_subset,
        tradeoff=0,
        threads=2
    )

    out_file = os.path.join(output_dir, f"grow_results_micom_{sample_id}.zip")
    res.save(out_file)
    print(f"Результат сохранён: {out_file}")

Обработка образца sample_1_10


Output()

Результат сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/grow_results_micom/grow_results_micom_sample_1_10.zip
Обработка образца sample_1_5


Output()

Результат сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/grow_results_micom/grow_results_micom_sample_1_5.zip
Обработка образца sample_1_6


...

Результат сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/grow_results_micom/grow_results_micom_sample_3_6.zip
Обработка образца sample_3_7


Output()

Результат сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/grow_results_micom/grow_results_micom_sample_3_7.zip
Обработка образца sample_3_8


Output()

Результат сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/grow_results_micom/grow_results_micom_sample_3_8.zip


In [ ]:
import os
import zipfile
import pandas as pd
import tempfile

input_dir = "/content/drive/MyDrive/path_to_file/grow_results_micom"
output_dir = "/content/drive/MyDrive/path_to_file/final_res_with_sample_id"
os.makedirs(output_dir, exist_ok=True)

all_annotations = []
all_exchanges = []
all_growth_rates = []

for zip_filename in os.listdir(input_dir):
    if not zip_filename.startswith("grow_results_micom_") or not zip_filename.endswith(".zip"):
        continue

    sample_id = zip_filename.replace("grow_results_micom_", "").replace(".zip", "")
    print(f"{zip_filename} -> sample_id = {sample_id}")

    zip_path = os.path.join(input_dir, zip_filename)

    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tmpdir)

        for root, dirs, files in os.walk(tmpdir):
            for file in files:
                if file.endswith(".csv"):
                    csv_path = os.path.join(root, file)
                    df = pd.read_csv(csv_path)

                    if 'sample_id' in df.columns:
                        df['sample_id'] = sample_id
                    else:
                        df.insert(0, 'sample_id', sample_id)

                    if file == "annotations.csv":
                        all_annotations.append(df)
                    elif file == "exchanges.csv":
                        all_exchanges.append(df)
                    elif file == "growth_rates.csv":
                        all_growth_rates.append(df)
                    else:
                        print(f"{file}")

if all_annotations:
    final_annotations = pd.concat(all_annotations, ignore_index=True)
    final_annotations.to_csv(os.path.join(output_dir, "annotations.csv"), index=False)
    print(f"{len(final_annotations)}")

if all_exchanges:
    final_exchanges = pd.concat(all_exchanges, ignore_index=True)
    final_exchanges.to_csv(os.path.join(output_dir, "exchanges.csv"), index=False)
    print(f"Сохранено exchanges.csv: {len(final_exchanges)} строк")

if all_growth_rates:
    final_growth_rates = pd.concat(all_growth_rates, ignore_index=True)
    final_growth_rates.to_csv(os.path.join(output_dir, "growth_rates.csv"), index=False)
    print(f"Сохранено growth_rates.csv: {len(final_growth_rates)} строк")

archive_path = os.path.join(output_dir, "final_results.zip")
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for csv_file in ["annotations.csv", "exchanges.csv", "growth_rates.csv"]:
        file_path = os.path.join(output_dir, csv_file)
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=csv_file)
print(f"{archive_path}")

Обработка: grow_results_micom_sample_1_8.zip -> sample_id = sample_1_8
Обработка: grow_results_micom_sample_1_9.zip -> sample_id = sample_1_9
Обработка: grow_results_micom_sample_2_2.zip -> sample_id = sample_2_2
Обработка: grow_results_micom_sample_2_3.zip -> sample_id = sample_2_3
Обработка: grow_results_micom_sample_3_2.zip -> sample_id = sample_3_2
Обработка: grow_results_micom_sample_3_3.zip -> sample_id = sample_3_3
Сохранено annotations.csv: 1940 строк
Сохранено exchanges.csv: 370 строк
Сохранено growth_rates.csv: 12 строк

Архив сохранён: /content/drive/MyDrive/sup_notebooks/final/dagitata/aerobic/final_res_with_sample_id/final_results.zip
Готово! Все три объединённые таблицы лежат в папке final_res_with_sample_id и упакованы в final_results.zip


In [ ]:
from micom.data import test_results
from micom.interaction import interactions

results = test_results()

In [ ]:
import zipfile
import pandas as pd
import os

zip_path = "/content/drive/MyDrive/path_to_file/final_res_with_sample_id/final_results.zip"
out_path = "/content/drive/MyDrive/path_to_file/final_res_with_sample_id/final_results_agitata.zip"

temp_dir = "/tmp/final_results_temp"
os.makedirs(temp_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(temp_dir)

annot_path = os.path.join(temp_dir, "annotations.csv")
df = pd.read_csv(annot_path)
if 'sample_id' in df.columns:
    df = df.drop(columns=['sample_id'])
    df.to_csv(annot_path, index=False)
    print("column sample_id removed from annotation.csv")
else:
    print("column sample_id not found")

with zipfile.ZipFile(out_path, 'w') as zf:
    for root, dirs, files in os.walk(temp_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, temp_dir)
            zf.write(full_path, arcname)

import shutil
shutil.rmtree(temp_dir)


results = results.load("/content/drive/MyDrive/path_to_file/final_res_with_sample_id/final_results_agitata.zip")


Столбец sample_id удалён из annotation.csv


In [ ]:
import pandas as pd
import numpy as np

def count_non_empty(row):
    return row.notna().sum()

results.annotations['_info_score'] = results.annotations.apply(count_non_empty, axis=1)

results.annotations = results.annotations.sort_values('_info_score', ascending=False)

results.annotations = results.annotations.drop_duplicates(subset=['metabolite'], keep='first') #, 'reaction'

results.annotations = results.annotations.drop(columns=['_info_score'])

results.annotations = results.annotations.reset_index(drop=True)

results.annotations

,metabolite,name,molecular_weight,C_number,N_number,seed.compound,kegg.compound,bigg.metabolite,metanetx.chemical,sbo,...,metacyc.compound,mnx,pubchem.compound,envipath,envipathm,kegg.drug,keggd,lipidmapsm,lipidmaps,slm
0,ac_m,Acetate-e,59.04402,2,0,cpd00029,C00033,ac,MNXM26,NaN,...,"['CPD-24335', 'ACET']",NaN,175,['5882df9c-dae1-4d80-a40e-db4724271456/compoun...,['5882df9c-dae1-4d80-a40e-db4724271456/compoun...,D00010,"['D00010', 'M_D00010']",LMFA01010002,LMFA01010002,449.0
1,ac_e,Acetate-e,59.04402,2,0,cpd00029,C00033,ac,MNXM26,NaN,...,"['CPD-24335', 'ACET']",NaN,175,['5882df9c-dae1-4d80-a40e-db4724271456/compoun...,['5882df9c-dae1-4d80-a40e-db4724271456/compoun...,D00010,"['D00010', 'M_D00010']",LMFA01010002,LMFA01010002,449.0
2,val__L_e,L-Valine-e,117.14634,5,1,"['cpd00156', 'cpd15141']","['C00183', 'C16436']",val__L,MNXM199,NaN,...,VAL,NaN,6287 6971018 88733505,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,D00039,"['M_D00039', 'D00039']",LMFA01100046,LMFA01100046,NaN
3,leu__L_m,L-Leucine-c,131.17292,6,1,"['cpd00107', 'cpd15143']","['C00123', 'C16439']",leu__L,MNXM1106761,NaN,...,LEU,NaN,6106 7045798,['650babc9-9d68-4b73-9332-11972ca26f7b/compoun...,['650babc9-9d68-4b73-9332-11972ca26f7b/compoun...,D00030,"['M_D00030', 'D00030']",LMFA01100048,LMFA01100048,NaN
4,val__L_m,L-Valine-e,117.14634,5,1,"['cpd00156', 'cpd15141']","['C00183', 'C16436']",val__L,MNXM199,NaN,...,VAL,NaN,6287 6971018 88733505,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,D00039,"['M_D00039', 'D00039']",LMFA01100046,LMFA01100046,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,clo2_e,Chlorite,67.45180,0,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
374,c2h6_m,ethane,30.06904,2,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
375,c2h6_e,ethane,30.06904,2,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
376,ham_m,hydroxylamine,33.02992,0,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import micom
from micom.interaction import interactions

ints = interactions(results, taxa="album")
# ints = ints.sort_values(by="flux").head()
ints

,sample_id,metabolite,focal,partner,class,flux,name,molecular_weight,C_number,N_number,...,metacyc.compound,mnx,pubchem.compound,envipath,envipathm,kegg.drug,keggd,lipidmapsm,lipidmaps,slm
0,sample_2_3,fe2_e,zr20,agitata,co-consumed,0.000002,Fe2-c,55.845000,0,0,...,FE+2,NaN,27284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sample_2_3,h_e,zr20,agitata,provided,3.113717,H-e,1.007940,0,0,...,PROTON,PROTON,1038,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sample_2_3,lac__L_e,zr20,agitata,provided,3.114707,L-Lactate-e,89.070000,3,0,...,"['L-LACTATE', 'CPD-24844']",NaN,5460161,650babc9-9d68-4b73-9332-11972ca26f7b/compound/...,650babc9-9d68-4b73-9332-11972ca26f7b/compound/...,NaN,NaN,NaN,NaN,NaN
3,sample_2_3,nh3_e,zr20,agitata,co-consumed,0.001220,NH3-e,18.038460,0,1,...,"['AMMONIA', 'AMMONIUM']",NaN,223 16741146,['d817c31c-d879-44c8-913d-2963d42f8203/compoun...,['d817c31c-d879-44c8-913d-2963d42f8203/compoun...,"['D02915', 'D02916']","['M_D02916', 'M_D02915', 'D02915', 'D02916']",NaN,NaN,NaN
4,sample_2_3,o2_e,zr20,agitata,received,9.360009,O2-e,31.998800,0,0,...,OXYGEN-MOLECULE,NaN,977,d817c31c-d879-44c8-913d-2963d42f8203/compound/...,d817c31c-d879-44c8-913d-2963d42f8203/compound/...,D00003,"['D00003', 'M_D00003']",NaN,NaN,NaN
5,sample_2_3,pi_e,zr20,agitata,co-consumed,0.000119,Phosphate-e,95.979301,0,0,...,"['CPD-20124', 'Isoprenoid-Derivatives', 'Pi', ...",NaN,3681305 57424078,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,['32de3cf4-e3e6-4168-956e-32fa5ddb0ce1/compoun...,D05467,"['M_D05467', 'D05467']",NaN,NaN,NaN
6,sample_2_4,ca2_e,zr20,agitata,co-consumed,0.000001,Ca2-e,40.078000,0,0,...,CA+2,NaN,271,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,sample_2_4,cu2_e,zr20,agitata,co-consumed,0.000001,Cu2-c,63.546000,0,0,...,NaN,NaN,23978,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,sample_2_4,fe2_e,zr20,agitata,co-consumed,0.000001,Fe2-c,55.845000,0,0,...,FE+2,NaN,27284,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,sample_2_4,h_e,zr20,agitata,provided,2.724670,H-e,1.007940,0,0,...,PROTON,PROTON,1038,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from micom.interaction import summarize_interactions

summary = summarize_interactions(ints)
summary.head()

,sample_id,focal,partner,class,flux,mass_flux,C_flux,N_flux,n_ints
0,sample_2_3,zr20,agitata,co-consumed,0.001341,0.000034,0.000000,0.001220,3
1,sample_2_3,zr20,agitata,provided,6.228424,0.280565,9.344121,0.000000,2
2,sample_2_3,zr20,agitata,received,9.360009,0.299509,0.000000,0.000000,1
3,sample_2_4,zr20,agitata,co-consumed,0.001702,0.000046,0.000000,0.001504,7
4,sample_2_4,zr20,agitata,provided,5.450935,0.245575,8.178796,0.000000,2


In [ ]:
full = interactions(results, taxa=None, threads=8)
full.shape

Output()

(414, 41)

In [ ]:
full_summary = summarize_interactions(full)
full_summary.shape

(90, 9)

In [ ]:
from micom.interaction import MES

scores = MES(results)
scores.head()

,metabolite,sample_id,MES,name,molecular_weight,C_number,N_number,seed.compound,kegg.compound,bigg.metabolite,...,metacyc.compound,mnx,pubchem.compound,envipath,envipathm,kegg.drug,keggd,lipidmapsm,lipidmaps,slm
0,4_Hydroxy_benzylalcohol_e,sample_3_2,0.0,4-Hydroxy-benzylalcohol-e,124.13722,7,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4_Hydroxy_benzylalcohol_e,sample_3_3,0.0,4-Hydroxy-benzylalcohol-e,124.13722,7,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4_Hydroxy_benzylalcohol_e,sample_3_4,0.0,4-Hydroxy-benzylalcohol-e,124.13722,7,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4_Hydroxy_benzylalcohol_e,sample_3_5,0.0,4-Hydroxy-benzylalcohol-e,124.13722,7,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4_Hydroxy_benzylalcohol_e,sample_3_6,0.0,4-Hydroxy-benzylalcohol-e,124.13722,7,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from micom.viz import plot_growth

pl = plot_growth(results, filename="/content/drive/MyDrive/path_to_file/growth_rates_st.html")

In [ ]:
from micom.viz import plot_exchanges_per_sample

pl = plot_exchanges_per_sample(results, filename="/content/drive/MyDrive/path_to_file/consumption_st.html")

In [ ]:
from micom.viz import plot_exchanges_per_taxon

pl = plot_exchanges_per_taxon(results, filename="/content/drive/MyDrive/path_to_file/niche_st.html")

In [ ]:
from micom.viz import plot_focal_interactions

pl = plot_focal_interactions(results, taxon='album', filename="/content/drive/MyDrive/path_to_file/interactions_album.html")

ValueError: The (cleaned) taxon name ob3 is not in the data set. Possible taxa are: agitata,mcaps,zr20,album.